In [ ]:
import numpy as np
import os
import xarray as xr
import pandas as pd
import sys

sys.path.append('..//')
from utils_mitgcm import open_mitgcm_ds_from_config
from utils_signal_processing import *
from utils_energy_analysis import *

import dask
from dask.distributed import Client
dask.config.set({
    'distributed.worker.memory.target': 0.6,  # fraction of memory to start spilling
    'distributed.worker.memory.spill': 0.7,   # fraction to spill to disk
    'distributed.worker.memory.pause': 0.8,   # fraction to pause worker
})

client = Client(processes=True, n_workers=20, threads_per_worker=1)

client.dashboard_link

ssh -L 8787:127.0.0.1:8787 leroquan@KBALL-LAKEN-L

In [ ]:
lake= 'neuchatel'
model = f'{lake}_2025'
mitgcm_config, ds = open_mitgcm_ds_from_config('..//config.json', model)

In [ ]:
folder_path = os.path.dirname(mitgcm_config['datapath'])
output_folder = os.path.join(folder_path, "seiche_analysis")
os.makedirs(output_folder, exist_ok=True)

# Load data

In [ ]:
grid_resolution = 100
ds['YC'] = np.arange(1, len(ds['YC'])+1) * grid_resolution - grid_resolution/2
ds['XC'] = np.arange(1, len(ds['XC'])+1) * grid_resolution - grid_resolution/2
ds['YG'] = np.arange(0, len(ds['YG'])) * grid_resolution
ds['XG'] = np.arange(0, len(ds['XG'])) * grid_resolution

u = ds.UVEL.chunk({'time':-1,'Z':1, 'XG':1, 'YC':1})
v = ds.VVEL.chunk({'time':-1,'Z':1, 'XC':1, 'YG':1})
w = ds.WVEL.chunk({'time':-1,'Zl':1, 'XC':1, 'YC':1})

u = ds.UVEL.chunk({'time':-1,'Z':1})#.sel(time=slice(pd.to_datetime("2025-08-01"), pd.to_datetime("2025-09-01")))
v = ds.VVEL.chunk({'time':-1,'Z':1})#.sel(time=slice(pd.to_datetime("2025-08-01"), pd.to_datetime("2025-09-01")))
w = ds.WVEL.chunk({'time':-1,'Zl':1})#.sel(time=slice(pd.to_datetime("2025-08-01"), pd.to_datetime("2025-09-01")))

# Defining cutoff for filtering

In [ ]:
cutoff1_hr = 45
cutoff2_hr = 30

In [ ]:
path_cutoff_folder = os.path.join(output_folder, f'{cutoff2_hr}_{cutoff1_hr}h')
os.makedirs(path_cutoff_folder, exist_ok=True)

# Filter timeseries

In [ ]:
import dask
import xarray as xr

In [ ]:
u_filtered_list = []
v_filtered_list = []
w_filtered_list = []

for xx in range(0, len(ds.XG)): #len(ds.XG)
    u = ds.UVEL.isel(XG=xx).chunk({'time': -1}).persist()
    v = ds.VVEL.isel(XC=xx).chunk({'time': -1}).persist()
    w = ds.WVEL.isel(XC=xx).chunk({'time': -1}).persist()

    period_low = cutoff1_hr * 3600
    period_high = cutoff2_hr * 3600
    _filter_kwargs = dict(
        btype="bandpass",
        time_dim="time",
        dt=3600,
        period_cutoff_low=period_low,
        period_cutoff_high=period_high,
        order=5,
    )

    u_f = filter_signal_xarray(u, **_filter_kwargs)
    v_f = filter_signal_xarray(v, **_filter_kwargs)
    w_f = filter_signal_xarray(w, **_filter_kwargs)

    u_filtered_temp, v_filtered_temp, w_filtered_temp = dask.persist(u_f, v_f, w_f)

    u_filtered_list.append(u_filtered_temp.expand_dims(XG=[ds["XG"].values[xx]]))
    v_filtered_list.append(v_filtered_temp.expand_dims(XC=[ds["XC"].values[xx]]))
    w_filtered_list.append(w_filtered_temp.expand_dims(XC=[ds["XC"].values[xx]]))

u_filtered = xr.concat(u_filtered_list, dim="XG")
v_filtered = xr.concat(v_filtered_list, dim="XC")
w_filtered = xr.concat(w_filtered_list, dim="XC")


# Compute total KE

In [ ]:
aligned_u = u_filtered.rename({'XG':'XC'})
aligned_u['XC'] = v_filtered['XC']

aligned_v = v_filtered.rename({'YG':'YC'})
aligned_v['YC'] = u_filtered['YC']

aligned_w = w_filtered.rename({'Zl':'Z'})
aligned_w['Z'] = v_filtered['Z']

In [ ]:
ke_tot = compute_ke(
    aligned_u,
    aligned_v,
    aligned_w,
    grid_resolution,
    grid_resolution,
    ds.drF)

In [ ]:
df_ke_tot = ke_tot.sum(dim=['XC','YC','Z']).to_dataframe(name='ke_mj_total')['ke_mj_total']

In [ ]:
df_ke_tot.reset_index().to_csv(os.path.join(output_folder, "ke_seiche.csv"))

In [ ]:
df_ke_tot.plot()

In [ ]:
(aligned_u.isel(XC=0, YC=40, Z=0)**2).plot()
(aligned_u.isel(XC=0, YC=40, Z=0)/20).plot()

In [ ]:
l_seg = len(u.time)
ke_fft = xr_compute_meanfft(aligned_u.isel(XC=0, YC=40, Z=0)**2, seg_length=l_seg)

In [ ]:
cutoff1 = 1/(cutoff1_hr * 3600)
cutoff2 = 1/(cutoff2_hr * 3600)

In [ ]:
fig,ax = plot_freq_spectrum(ke_fft, 'U', depth=0, l_segm=l_seg, y_lim_min=1e-18, x_lim_min=0.01e-4, fontsize=10)
ax.axvline(x = cutoff1, linestyle="--", color="k",label="cutoff1")
ax.axvline(x = cutoff2, linestyle="--", color="k",label="cutoff2")
ax.legend()

# Save results

In [ ]:
u_filtered['drF'] = u_filtered['drF'].astype('<f8')
v_filtered['drF'] = v_filtered['drF'].astype('<f8')

In [ ]:
coords_to_drop = ["dyG", "dxC", "dxG", "dyC", "rAs", "hFacS", "rAw", "PHrefC", "hFacW", "rhoRef", "rA", "Depth", "dxF", "dyF"]

In [ ]:
u_filtered.drop_vars(coords_to_drop, errors='ignore').to_zarr(os.path.join(path_cutoff_folder, rf"u_filtered_august2025_{cutoff2_hr}-{cutoff1_hr}h.zarr"), mode='w')
v_filtered.drop_vars(coords_to_drop, errors='ignore').to_zarr(os.path.join(path_cutoff_folder, rf"v_filtered_august2025_{cutoff2_hr}-{cutoff1_hr}h.zarr"), mode='w')
w_filtered.drop_vars(coords_to_drop, errors='ignore').to_zarr(os.path.join(path_cutoff_folder, rf"w_filtered_august2025_{cutoff2_hr}-{cutoff1_hr}h.zarr"), mode='w')

In [ ]:
u_filtered = xr.open_zarr(os.path.join(path_cutoff_folder, rf"u_filtered_august2025_{cutoff2_hr}-{cutoff1_hr}h.zarr"))
v_filtered = xr.open_zarr(os.path.join(path_cutoff_folder, rf"v_filtered_august2025_{cutoff2_hr}-{cutoff1_hr}h.zarr"))
w_filtered = xr.open_zarr(os.path.join(path_cutoff_folder, rf"w_filtered_august2025_{cutoff2_hr}-{cutoff1_hr}h.zarr"))


# Save residual

In [ ]:
u_residual = (ds.UVEL - u_filtered)
v_residual = (ds.VVEL - v_filtered)
w_residual = (ds.WVEL - w_filtered)

In [ ]:
u_residual['drF'] = u_residual['drF'].astype('<f8')
v_residual['drF'] = v_residual['drF'].astype('<f8')

u_residual.drop_vars(coords_to_drop, errors='ignore').to_zarr(os.path.join(path_cutoff_folder, rf"u_residual_{cutoff2_hr}-{cutoff1_hr}h.zarr"), mode='w')
v_residual.drop_vars(coords_to_drop, errors='ignore').to_zarr(os.path.join(path_cutoff_folder, rf"v_residual_{cutoff2_hr}-{cutoff1_hr}h.zarr"), mode='w')
w_residual.drop_vars(coords_to_drop, errors='ignore').to_zarr(os.path.join(path_cutoff_folder, rf"w_residual_{cutoff2_hr}-{cutoff1_hr}h.zarr"), mode='w')

In [ ]:
u_residual.isel(time=i_time, Z=0).UVEL.shape

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
i_time=24*14
plt.figure(figsize=(15, 5))
uplot=u_residual.isel(time=i_time, Z=0).UVEL
vplot=v_residual.isel(time=i_time, Z=0).VVEL
xgrid, ygrid = np.meshgrid(range(uplot.shape[1]), range(uplot.shape[0]))
plt.streamplot(xgrid, ygrid, uplot, vplot,
               density=4, color='black', linewidth=0.5,
               arrowsize=0.7, arrowstyle='->')
plt.imshow(uplot)
plt.colorbar()
plt.text(0.02, 0.98, f'{uplot.time.values}', transform=plt.gca().transAxes, ha='left', va='top')

In [ ]:
i_time=24*14
plt.figure(figsize=(15, 5))
uplot=u_filtered.isel(time=i_time, Z=0).UVEL.T
vplot=v_filtered.isel(time=i_time, Z=0).VVEL.T
xgrid, ygrid = np.meshgrid(range(uplot.shape[1]), range(uplot.shape[0]))
plt.streamplot(xgrid, ygrid, uplot, vplot,
               density=4, color='black', linewidth=0.5,
               arrowsize=0.7, arrowstyle='->')
plt.imshow(uplot)
plt.colorbar()
plt.text(0.02, 0.98, f'{uplot.time.values}', transform=plt.gca().transAxes, ha='left', va='top')

In [ ]:
i_time=24*14
plt.figure(figsize=(15, 5))
uplot=ds.UVEL.isel(time=i_time, Z=0)
vplot=ds.VVEL.isel(time=i_time, Z=0)
xgrid, ygrid = np.meshgrid(range(uplot.shape[1]), range(uplot.shape[0]))
plt.streamplot(xgrid, ygrid, uplot, vplot,
               density=4, color='black', linewidth=0.5,
               arrowsize=0.7, arrowstyle='->')
plt.imshow(uplot)
plt.colorbar()
plt.text(0.02, 0.98, f'{uplot.time.values}', transform=plt.gca().transAxes, ha='left', va='top')